In [1]:
import argparse
import os
import time
import shutil

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torch.backends.cudnn as cudnn

from tensorboardX import SummaryWriter      

import torchvision
import torchvision.transforms as transforms

import torch.nn.utils.prune as prune

from models import *

global best_prec
use_gpu = torch.cuda.is_available()
print('=> Building model...')
    
    
batch_size = 128
model_name = "VGG16_quant"
model = VGG16_quant()
print(model)

normalize = transforms.Normalize(mean=[0.491, 0.482, 0.447], std=[0.247, 0.243, 0.262])


train_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        normalize,
    ]))
trainloader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)


test_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=False,
    download=True,
    transform=transforms.Compose([
        transforms.ToTensor(),
        normalize,
    ]))

testloader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)


print_freq = 100 # every 100 batches, accuracy printed. Here, each batch includes "batch_size" data points
# CIFAR10 has 50,000 training data, and 10,000 validation data.

def train(trainloader, model, criterion, optimizer, epoch):
    batch_time = AverageMeter()
    data_time = AverageMeter()
    losses = AverageMeter()
    top1 = AverageMeter()

    model.train()

    end = time.time()
    for i, (input, target) in enumerate(trainloader):
        # measure data loading time
        data_time.update(time.time() - end)

        input, target = input.cuda(), target.cuda()

        # compute output
        output = model(input)
        loss = criterion(output, target)

        # measure accuracy and record loss
        prec = accuracy(output, target)[0]
        losses.update(loss.item(), input.size(0))
        top1.update(prec.item(), input.size(0))

        # compute gradient and do SGD step
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # measure elapsed time
        batch_time.update(time.time() - end)
        end = time.time()


        if i % print_freq == 0:
            print('Epoch: [{0}][{1}/{2}]\t'
                  'Time {batch_time.val:.3f} ({batch_time.avg:.3f})\t'
                  'Data {data_time.val:.3f} ({data_time.avg:.3f})\t'
                  'Loss {loss.val:.4f} ({loss.avg:.4f})\t'
                  'Prec {top1.val:.3f}% ({top1.avg:.3f}%)'.format(
                   epoch, i, len(trainloader), batch_time=batch_time,
                   data_time=data_time, loss=losses, top1=top1))

            

def validate(val_loader, model, criterion ):
    batch_time = AverageMeter()
    losses = AverageMeter()
    top1 = AverageMeter()

    # switch to evaluate mode
    model.eval()

    end = time.time()
    with torch.no_grad():
        for i, (input, target) in enumerate(val_loader):
         
            input, target = input.cuda(), target.cuda()

            # compute output
            output = model(input)
            loss = criterion(output, target)

            # measure accuracy and record loss
            prec = accuracy(output, target)[0]
            losses.update(loss.item(), input.size(0))
            top1.update(prec.item(), input.size(0))

            # measure elapsed time
            batch_time.update(time.time() - end)
            end = time.time()

            if i % print_freq == 0:  # This line shows how frequently print out the status. e.g., i%5 => every 5 batch, prints out
                print('Test: [{0}/{1}]\t'
                  'Time {batch_time.val:.3f} ({batch_time.avg:.3f})\t'
                  'Loss {loss.val:.4f} ({loss.avg:.4f})\t'
                  'Prec {top1.val:.3f}% ({top1.avg:.3f}%)'.format(
                   i, len(val_loader), batch_time=batch_time, loss=losses,
                   top1=top1))

    print(' * Prec {top1.avg:.3f}% '.format(top1=top1))
    return top1.avg


def accuracy(output, target, topk=(1,)):
    """Computes the precision@k for the specified values of k"""
    maxk = max(topk)
    batch_size = target.size(0)

    _, pred = output.topk(maxk, 1, True, True)
    pred = pred.t()
    correct = pred.eq(target.view(1, -1).expand_as(pred))

    res = []
    for k in topk:
        correct_k = correct[:k].view(-1).float().sum(0)
        res.append(correct_k.mul_(100.0 / batch_size))
    return res


class AverageMeter(object):
    """Computes and stores the average and current value"""
    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count

        
def save_checkpoint(state, is_best, fdir):
    filepath = os.path.join(fdir, 'checkpoint.pth')
    torch.save(state, filepath)
    if is_best:
        shutil.copyfile(filepath, os.path.join(fdir, 'model_best.pth.tar'))


def adjust_learning_rate(optimizer, epoch):
    """For resnet, the lr starts from 0.1, and is divided by 10 at 80 and 120 epochs"""
    adjust_list = [150, 225]
    if epoch in adjust_list:
        for param_group in optimizer.param_groups:
            param_group['lr'] = param_group['lr'] * 0.1        

#model = nn.DataParallel(model).cuda()
#all_params = checkpoint['state_dict']
#model.load_state_dict(all_params, strict=False)
#criterion = nn.CrossEntropyLoss().cuda()
#validate(testloader, model, criterion)

=> Building model...
VGG_quant(
  (features): Sequential(
    (0): QuantConv2d(
      3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False
      (weight_quant): weight_quantize_fn()
    )
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): QuantConv2d(
      64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False
      (weight_quant): weight_quantize_fn()
    )
    (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU(inplace=True)
    (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (7): QuantConv2d(
      64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False
      (weight_quant): weight_quantize_fn()
    )
    (8): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (9): ReLU(inplace=True)
    (10): QuantConv2d(
      128, 128, kernel_size=(3, 3), stride

In [2]:
# HW

#  1. Load your saved model and validate
#  2. Replace your model's all the Conv's weight with quantized weight
#  3. Apply reasonable alpha
#  4. Then, try to multiple bit precisions and draw graph of bit precision vs. accuracy

In [3]:
PATH = "result/VGG16_quant/model_best.pth.tar"
checkpoint = torch.load(PATH)
model.load_state_dict(checkpoint['state_dict'])
device = torch.device("cuda") 

model.cuda()
model.eval()

test_loss = 0
correct = 0

with torch.no_grad():
    for data, target in testloader:
        data, target = data.to(device), target.to(device) # loading to GPU
        output = model(data)
        pred = output.argmax(dim=1, keepdim=True)  
        correct += pred.eq(target.view_as(pred)).sum().item()

test_loss /= len(testloader.dataset)

print('\nTest set: Accuracy: {}/{} ({:.0f}%)\n'.format(
        correct, len(testloader.dataset),
        100. * correct / len(testloader.dataset)))


Test set: Accuracy: 9279/10000 (93%)



In [4]:
#### Prune all the QuantConv2D layers' 80% weights with 1) unstructured, and 2) structured manner.

In [6]:
i = 0
for layer in model.modules():
    i = i+1
    if isinstance(layer, QuantConv2d):
        prune.l1_unstructured(layer, name="weight", amount=0.8)

In [7]:
print(list(model.features[40].named_parameters())) # check whether there is mask, weight_org, ...
print(model.features[40].weight) # check whether there are many zeros

[('act_alpha', Parameter containing:
tensor(0.6882, device='cuda:0', requires_grad=True)), ('weight_q', Parameter containing:
tensor([[[[-0.0000, -0.0000, -0.0000],
          [-0.0000, -0.0000, -0.0000],
          [-0.0000, -0.0000, -0.0000]],

         [[-0.0000, -0.0000, -0.0000],
          [-0.0000, -0.0000, -0.0000],
          [-0.0000, -0.0000, -0.0000]],

         [[-0.0000, -0.0000, -0.0000],
          [-0.0000, -0.0000, -0.0000],
          [-0.0000, -0.0000, -0.0000]],

         ...,

         [[-0.0000, -0.0000, -0.0000],
          [-0.0000, -0.0000, -0.0000],
          [-0.0000, -0.0000, -0.0000]],

         [[-0.0000, -0.0000, -0.0000],
          [-0.0000, -0.0000, -0.0000],
          [-0.0000, -0.0000, -0.0000]],

         [[-0.0000, -0.0000, -0.0000],
          [-0.0000, -0.0000, -0.0000],
          [-0.0000, -0.0000, -0.0000]]],


        [[[-0.0000, -0.0000, -0.0000],
          [-0.0000, -0.5025, -0.0000],
          [ 0.0000,  0.5025, -0.0000]],

         [[-0.5025, -0.5

In [8]:
### Check sparsity ###
mask1 = model.features[40].weight_mask
sparsity_mask1 = (mask1 == 0).sum() / mask1.nelement()

print("Sparsity level: ", sparsity_mask1)

Sparsity level:  tensor(0.8000, device='cuda:0')


In [9]:
model.cuda()
model.eval()
test_loss = 0
correct = 0
with torch.no_grad():
    for data, target in testloader:
        data, target = data.to(device), target.to(device) # loading to GPU
        output = model(data)
        pred = output.argmax(dim=1, keepdim=True)  
        correct += pred.eq(target.view_as(pred)).sum().item()

test_loss /= len(testloader.dataset)

print('\nTest set: Accuracy: {}/{} ({:.0f}%)\n'.format(
        correct, len(testloader.dataset),
        100. * correct / len(testloader.dataset)))


Test set: Accuracy: 1003/10000 (10%)



In [10]:
lr = 1e-1
weight_decay = 1e-4
epochs = 100
best_prec = 0

#model = nn.DataParallel(model).cuda()
model.cuda()
criterion = nn.CrossEntropyLoss().cuda()
optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=weight_decay)
#cudnn.benchmark = True

if not os.path.exists('result'):
    os.makedirs('result')
fdir = 'result/'+'pruning'+str(model_name)
if not os.path.exists(fdir):
    os.makedirs(fdir)


for epoch in range(0, epochs):
    adjust_learning_rate(optimizer, epoch)

    train(trainloader, model, criterion, optimizer, epoch)

    # evaluate on test set
    print("Validation starts")
    prec = validate(testloader, model, criterion)

    # remember best precision and save checkpoint
    is_best = prec > best_prec
    best_prec = max(prec,best_prec)
    print('best acc: {:1f}'.format(best_prec))
    save_checkpoint({
        'epoch': epoch + 1,
        'state_dict': model.state_dict(),
        'best_prec': best_prec,
        'optimizer': optimizer.state_dict(),
    }, is_best, fdir)

Epoch: [0][0/391]	Time 0.302 (0.302)	Data 0.109 (0.109)	Loss 2.7144 (2.7144)	Prec 53.125% (53.125%)
Epoch: [0][100/391]	Time 0.052 (0.056)	Data 0.002 (0.003)	Loss 0.2513 (0.4470)	Prec 90.625% (85.961%)
Epoch: [0][200/391]	Time 0.054 (0.054)	Data 0.002 (0.002)	Loss 0.2806 (0.3418)	Prec 89.062% (89.004%)
Epoch: [0][300/391]	Time 0.053 (0.054)	Data 0.002 (0.002)	Loss 0.3157 (0.3003)	Prec 92.188% (90.147%)
Validation starts
Test: [0/79]	Time 0.124 (0.124)	Loss 0.3016 (0.3016)	Prec 92.969% (92.969%)
 * Prec 88.390% 
best acc: 88.390000
Epoch: [1][0/391]	Time 0.189 (0.189)	Data 0.144 (0.144)	Loss 0.2100 (0.2100)	Prec 92.969% (92.969%)
Epoch: [1][100/391]	Time 0.054 (0.055)	Data 0.001 (0.003)	Loss 0.1983 (0.1628)	Prec 91.406% (94.438%)
Epoch: [1][200/391]	Time 0.053 (0.054)	Data 0.002 (0.002)	Loss 0.2095 (0.1601)	Prec 92.188% (94.508%)
Epoch: [1][300/391]	Time 0.050 (0.054)	Data 0.002 (0.002)	Loss 0.1325 (0.1575)	Prec 93.750% (94.627%)
Validation starts
Test: [0/79]	Time 0.117 (0.117)	Loss 0.

KeyboardInterrupt: 

In [11]:
PATH = "result/pruningVGG16_quant/model_best.pth.tar"
checkpoint = torch.load(PATH)
model.load_state_dict(checkpoint['state_dict']) 

model.cuda()

model.eval()
test_loss = 0
correct = 0
with torch.no_grad():
    for data, target in testloader:
        data, target = data.to(device), target.to(device) # loading to GPU
        output = model(data)
        pred = output.argmax(dim=1, keepdim=True)  
        correct += pred.eq(target.view_as(pred)).sum().item()

test_loss /= len(testloader.dataset)

print('\nTest set: Accuracy: {}/{} ({:.0f}%)\n'.format(
        correct, len(testloader.dataset),
        100. * correct / len(testloader.dataset)))


Test set: Accuracy: 8998/10000 (90%)



In [12]:
PATH = "result/VGG16_quant/model_best.pth.tar"
checkpoint = torch.load(PATH)
model = VGG16_quant()

model.load_state_dict(checkpoint['state_dict'])
model.cuda()
i = 0
for layer in model.modules():
    i = i+1
    if isinstance(layer, QuantConv2d):
        prune.ln_structured(layer, name="weight", amount=0.8, n = 1, dim = 0)

In [ ]:
print(list(model.features[40].named_parameters())) # check whether there is mask, weight_org, ...
print(model.features[40].weight) # check whether there are many zeros

In [ ]:
### Check sparsity ###
mask1 = model.features[40].weight_mask
sparsity_mask1 = (mask1 == 0).sum() / mask1.nelement()

print("Sparsity level: ", sparsity_mask1)

In [ ]:
model.eval()
test_loss = 0
correct = 0
with torch.no_grad():
    for data, target in testloader:
        data, target = data.to(device), target.to(device) # loading to GPU
        output = model(data)
        pred = output.argmax(dim=1, keepdim=True)  
        correct += pred.eq(target.view_as(pred)).sum().item()

test_loss /= len(testloader.dataset)

print('\nTest set: Accuracy: {}/{} ({:.0f}%)\n'.format(
        correct, len(testloader.dataset),
        100. * correct / len(testloader.dataset)))

In [ ]:
lr = 1e-1
weight_decay = 1e-4
epochs = 50
best_prec = 0

#model = nn.DataParallel(model).cuda()
model.cuda()
criterion = nn.CrossEntropyLoss().cuda()
optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=weight_decay)
#cudnn.benchmark = True

if not os.path.exists('result'):
    os.makedirs('result')
fdir = 'result/'+'pruning'+str(model_name)
if not os.path.exists(fdir):
    os.makedirs(fdir)


for epoch in range(0, epochs):
    adjust_learning_rate(optimizer, epoch)

    train(trainloader, model, criterion, optimizer, epoch)

    # evaluate on test set
    print("Validation starts")
    prec = validate(testloader, model, criterion)

    # remember best precision and save checkpoint
    is_best = prec > best_prec
    best_prec = max(prec,best_prec)
    print('best acc: {:1f}'.format(best_prec))
    save_checkpoint({
        'epoch': epoch + 1,
        'state_dict': model.state_dict(),
        'best_prec': best_prec,
        'optimizer': optimizer.state_dict(),
    }, is_best, fdir)

In [13]:
PATH = "result/VGG16_quant/model_best.pth.tar"
checkpoint = torch.load(PATH, map_location="cuda")
model = VGG16_quant()
model.load_state_dict(checkpoint['state_dict'])
model.cuda()
model.eval()

structured_amount = 0.5 
final_target_sparsity = 0.8

params_to_prune = []

# for layer in model.modules():
#     if isinstance(layer, QuantConv2d):
#         prune.ln_structured(
#             layer,
#             name="weight",
#             amount=structured_amount,
#             n=1,
#             dim=0,
#         )
#         params_to_prune.append((layer, "weight"))

# s1 = structured_amount
# S  = final_target_sparsity

# if s1 < S:
#     extra_unstruct = 1.0 - (1.0 - S) / (1.0 - s1)
# else:
#     extra_unstruct = 0.0
# prune.global_unstructured(
#     params_to_prune,
#     pruning_method=prune.L1Unstructured,
#     amount=extra_unstruct,  # 0.2
# )
for layer in model.modules():
    if isinstance(layer, QuantConv2d):
        prune.ln_structured(
            layer,
            name="weight",
            amount=structured_amount,
            n=1,
            dim=0,                   
        )
        w = layer.weight.detach()
        total = w.numel()
        zeros = (w == 0).sum().item()
        S_cur = zeros / total

        if S_cur >= final_target_sparsity:
            print(f"skip layer (already sparse): S_cur={S_cur:.3f}")
            continue

        S_t = final_target_sparsity
        extra = 1.0 - (1.0 - S_t) / (1.0 - S_cur)


        extra = max(0.0, min(1.0, extra))

        prune.l1_unstructured(
            layer,
            name="weight",
            amount=extra,
        )

# for layer in model.modules():
#     if isinstance(layer, QuantConv2d):
        

# for layer in model.modules():
#     if isinstance(layer, QuantConv2d):
#         if hasattr(layer, "weight_orig"):   # 有被 prune 過才會有這個屬性
#             prune.remove(layer, "weight")

In [14]:
def total_model_sparsity(model):
    total_elems = 0
    zero_elems = 0

    for m in model.modules():
        if isinstance(m, QuantConv2d):
            w = m.weight.data 
            total_elems += w.numel()
            zero_elems += (w == 0).sum().item()

    return zero_elems / total_elems

print("Model Sparsity =", total_model_sparsity(model))


Model Sparsity = 0.7999999864042358


In [15]:
model.eval()
test_loss = 0
correct = 0
with torch.no_grad():
    for data, target in testloader:
        data, target = data.to(device), target.to(device) # loading to GPU
        output = model(data)
        pred = output.argmax(dim=1, keepdim=True)  
        correct += pred.eq(target.view_as(pred)).sum().item()

test_loss /= len(testloader.dataset)

print('\nTest set: Accuracy: {}/{} ({:.0f}%)\n'.format(
        correct, len(testloader.dataset),
        100. * correct / len(testloader.dataset)))


Test set: Accuracy: 1000/10000 (10%)



In [16]:
lr = 1e-1
weight_decay = 1e-4
epochs = 50
best_prec = 0

#model = nn.DataParallel(model).cuda()
model.cuda()
criterion = nn.CrossEntropyLoss().cuda()
optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=weight_decay)
#cudnn.benchmark = True

if not os.path.exists('result'):
    os.makedirs('result')
fdir = 'result/'+'pruning'+str(model_name)
if not os.path.exists(fdir):
    os.makedirs(fdir)


for epoch in range(0, epochs):
    adjust_learning_rate(optimizer, epoch)

    train(trainloader, model, criterion, optimizer, epoch)

    # evaluate on test set
    print("Validation starts")
    prec = validate(testloader, model, criterion)

    # remember best precision and save checkpoint
    is_best = prec > best_prec
    best_prec = max(prec,best_prec)
    print('best acc: {:1f}'.format(best_prec))
    save_checkpoint({
        'epoch': epoch + 1,
        'state_dict': model.state_dict(),
        'best_prec': best_prec,
        'optimizer': optimizer.state_dict(),
    }, is_best, fdir)

Epoch: [0][0/391]	Time 0.161 (0.161)	Data 0.111 (0.111)	Loss 4.7153 (4.7153)	Prec 29.688% (29.688%)
Epoch: [0][100/391]	Time 0.053 (0.054)	Data 0.001 (0.003)	Loss 0.5078 (1.1068)	Prec 85.156% (65.640%)
Epoch: [0][200/391]	Time 0.053 (0.054)	Data 0.001 (0.002)	Loss 0.4099 (0.8319)	Prec 87.500% (73.488%)
Epoch: [0][300/391]	Time 0.053 (0.054)	Data 0.001 (0.002)	Loss 0.4423 (0.7104)	Prec 82.812% (77.056%)
Validation starts
Test: [0/79]	Time 0.131 (0.131)	Loss 0.4531 (0.4531)	Prec 86.719% (86.719%)
 * Prec 83.610% 
best acc: 83.610000
Epoch: [1][0/391]	Time 0.190 (0.190)	Data 0.154 (0.154)	Loss 0.3431 (0.3431)	Prec 86.719% (86.719%)
Epoch: [1][100/391]	Time 0.052 (0.055)	Data 0.001 (0.003)	Loss 0.4089 (0.3842)	Prec 87.500% (87.152%)
Epoch: [1][200/391]	Time 0.053 (0.054)	Data 0.001 (0.002)	Loss 0.3045 (0.3772)	Prec 89.062% (87.271%)
Epoch: [1][300/391]	Time 0.053 (0.054)	Data 0.001 (0.002)	Loss 0.2780 (0.3695)	Prec 90.625% (87.458%)
Validation starts
Test: [0/79]	Time 0.123 (0.123)	Loss 0.

In [17]:
model.eval()
test_loss = 0
correct = 0
with torch.no_grad():
    for data, target in testloader:
        data, target = data.to(device), target.to(device) # loading to GPU
        output = model(data)
        pred = output.argmax(dim=1, keepdim=True)  
        correct += pred.eq(target.view_as(pred)).sum().item()

test_loss /= len(testloader.dataset)

print('\nTest set: Accuracy: {}/{} ({:.0f}%)\n'.format(
        correct, len(testloader.dataset),
        100. * correct / len(testloader.dataset)))


Test set: Accuracy: 8687/10000 (87%)



In [ ]:
### Check sparsity ###
mask1 = model.features[40].weight_mask
sparsity_mask1 = (mask1 == 0).sum() / mask1.nelement()

print("Sparsity level: ", sparsity_mask1)